In [ ]:
pip install selenium

In [14]:
import csv
import os
import time
from datetime import date
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException

# Inicialize o WebDriver
opções = webdriver.ChromeOptions()
opções.add_argument('--headless')
opções.add_argument('--no-sandbox')
opções.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opções)

# Função para navegar até uma página específica
def navegar_para_pagina(url):
    driver.get(url)

# Função para rolar até o final da página
def rolar_ate_final():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Dicionário de URLs por categoria
urls = {
    'chuveiros-e-aquecedores': 'https://www.joli.com.br/chuveiros-e-aquecedores?page=1',
    'bomboniere': 'https://www.joli.com.br/bomboniere?page=1',
    'banheiros-cozinhas-e-areas-de-servico': 'https://www.joli.com.br/banheiros-cozinhas-e-areas-de-servico?page=1'
    # Adicione mais categorias e URLs conforme necessário
}

# Encontre os produtos
for categoria, url in urls.items():
    produtos = []
    produtos_carregados = set()  # Usaremos um set para rastrear produtos únicos
    pagina_atual = 1
    tempo_espera_inicial = 2
    incremento_tempo_espera = 2
    tempo_espera_maximo = 10

    print(f"\033[94mIniciando a coleta de produtos da categoria: {categoria}\033[0m")

    while True:
        try:
            # Navegue até a página atual
            url_com_pagina = f"{url.split('?')[0]}?page={pagina_atual}"
            navegar_para_pagina(url_com_pagina)

            # Rolagem e espera para garantir carregamento dos produtos
            while True:
                rolar_ate_final()
                time.sleep(tempo_espera_inicial)
                try:
                    botao_carregar_mais = driver.find_element(By.CSS_SELECTOR, 'footer')
                    if botao_carregar_mais.is_displayed():
                        continue
                except NoSuchElementException:
                    break

            # Aguarde até que os produtos sejam carregados
            produtos_atual = WebDriverWait(driver, tempo_espera_inicial).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem'))
            )

            # Adicione os produtos à lista
            novos_produtos = False
            for produto in produtos_atual:
                try:
                    # Coletar informações do produto
                    nome_empresa = "Joli"
                    data_coleta = date.today().strftime("%Y-%m-%d")
                    codigo_produto_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-brandName')
                    codigo_produto = codigo_produto_element.get_attribute('textContent').strip() if codigo_produto_element else "0"
                    imagem_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-image')
                    imagem_url = imagem_element.get_attribute('src')
                    nome_produto_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer')
                    nome_produto = nome_produto_element.get_attribute('textContent').strip()
                    moeda_element = produto.find_element(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price')
                    moeda = moeda_element.get_attribute('textContent').strip()
                    url_produto = produto.find_element(By.CSS_SELECTOR, 'a').get_attribute('href')

                    # Montar a linha de dados do produto
                    linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

                    # Adicionar à lista de produtos se ainda não estiver na lista
                    if nome_produto not in produtos_carregados:
                        produtos.append(linha_produto)
                        produtos_carregados.add(nome_produto)
                        novos_produtos = True

                except StaleElementReferenceException:
                    # Se o elemento não estiver mais presente, ignore-o
                    continue

            print(f"\033[93mTotal de produtos coletados de {categoria} até agora: {len(produtos)}\033[0m")

            if not novos_produtos:
                print(f"\033[93mNão há mais novos produtos para carregar de {categoria}.\033[0m")
                break

            # Aumentar o tempo de espera para a próxima iteração
            tempo_espera_inicial += incremento_tempo_espera
            if tempo_espera_inicial > tempo_espera_maximo:
                tempo_espera_inicial = tempo_espera_maximo

            # Avance para a próxima página
            pagina_atual += 1

        except TimeoutException:
            # Se não encontrar mais produtos, saia do loop
            print(f"\033[91mNão foi possível encontrar mais produtos em {categoria}.\033[0m")
            break

    # Salvar os produtos em um arquivo CSV com nome da categoria
    nome_arquivo = os.path.join(output_folder, f'produtos_{categoria}.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])

        for produto in produtos:
            escritor_csv.writerow(produto)

    print(f"\033[92mOs produtos da categoria {categoria} foram salvos em: {nome_arquivo}\033[0m")

# Feche o driver
driver.quit()


Iniciando a coleta de produtos da categoria: chuveiros-e-aquecedores
Total de produtos coletados de chuveiros-e-aquecedores até agora: 6
Total de produtos coletados de chuveiros-e-aquecedores até agora: 12
Total de produtos coletados de chuveiros-e-aquecedores até agora: 18
Total de produtos coletados de chuveiros-e-aquecedores até agora: 24
Total de produtos coletados de chuveiros-e-aquecedores até agora: 30
Total de produtos coletados de chuveiros-e-aquecedores até agora: 36
Total de produtos coletados de chuveiros-e-aquecedores até agora: 42
Total de produtos coletados de chuveiros-e-aquecedores até agora: 48
Total de produtos coletados de chuveiros-e-aquecedores até agora: 54
Total de produtos coletados de chuveiros-e-aquecedores até agora: 60
Total de produtos coletados de chuveiros-e-aquecedores até agora: 66
Total de produtos coletados de chuveiros-e-aquecedores até agora: 72
Total de produtos coletados de chuveiros-e-aquecedores até agora: 78
Total de produtos coletados de chuv